In [1]:
#some additional files i.e batters csv extractor code, bowler csv extractor code, build db extractor
#Note: while running this files make sure directory eg: DATA_DIRECTORY must be folder which has the json files 
# Recommended: always use .py files and then run for smoother run. running on notebooks can cause memory issues which may crash the jupyter kernel

In [ ]:
#batting csv file extractor
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

DATA_DIRECTORY = r"IPL"
MIN_MATCHES_REQUIRED = 5
MIN_RUNS_REQUIRED = 100

TEAM_NAME_MAPPING = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Deccan Chargers": "Sunrisers Hyderabad"
}

def standardize_team_name(team_name):

    return TEAM_NAME_MAPPING.get(team_name, team_name)

def parse_single_match(file_path):
    
    with open(file_path, encoding="utf-8") as file:
        match_data = json.load(file)

    
    match_info = match_data.get("info", {})
    match_dates = match_info.get("dates", ["1900-01-01"])
    match_year = int(str(match_dates[0])[:4])
    
    
    competing_teams = [standardize_team_name(team) for team in match_info.get("teams", [])]
    
    
    event_info = match_info.get("event", {})
    stage_name = event_info.get("stage", "").lower() if isinstance(event_info, dict) else ""
    is_knockout_match = any(keyword in stage_name for keyword in ["final", "semi", "playoff", "eliminator", "qualifier"])

    delivery_records = []

    for inning in match_data.get("innings", []):
        batting_team = standardize_team_name(inning.get("team", ""))
        
        bowling_team = ""
        for team in competing_teams:
            if team != batting_team:
                bowling_team = team
                break


        balls_faced_by_batter = defaultdict(int)
        batting_order_registry = {}
        next_batting_position = 1

        for over_data in inning.get("overs", []):
            over_number = over_data.get("over", 0)  

           
            if over_number <= 5:
                over_phase = "powerplay"
            elif over_number <= 14:
                over_phase = "middle"
            else:
                over_phase = "death"

            for delivery in over_data.get("deliveries", []):
                batter_name = delivery.get("batter", "")
                runs_scored_by_batter = delivery.get("runs", {}).get("batter", 0)
                
                
                extras_dict = delivery.get("extras", {})
                is_wide_ball = "wides" in extras_dict
                
                
                is_dismissed = False
                for wicket in delivery.get("wickets", []):
                    if wicket.get("player_out") == batter_name:
                        is_dismissed = True
                        break

                
                if batter_name not in batting_order_registry:
                    batting_order_registry[batter_name] = next_batting_position
                    next_batting_position += 1

                
                if not is_wide_ball:
                    balls_faced_by_batter[batter_name] += 1

                
                current_balls_faced = max(1, balls_faced_by_batter[batter_name])
                if current_balls_faced <= 10:
                    ball_phase = "phase_1_10"
                elif current_balls_faced <= 40:
                    ball_phase = "phase_10_40"
                else:
                    ball_phase = "phase_40plus"

                delivery_records.append({
                    "match_id": file_path.stem,
                    "year": match_year,
                    "batting_team": batting_team,
                    "opposition": bowling_team,
                    "batter": batter_name,
                    "runs": runs_scored_by_batter,
                    "is_wide": is_wide_ball,
                    "dismissed": is_dismissed,
                    "over": over_number,
                    "ball_phase": ball_phase,
                    "over_phase": over_phase,
                    "batting_position": batting_order_registry[batter_name],
                    "is_knockout": is_knockout_match,
                })

    return delivery_records


print("Reading and parsing JSON data profiles...")
all_match_files = list(Path(DATA_DIRECTORY).glob("*.json"))
raw_deliveries_list = []

for file_path in all_match_files:
    try:
        parsed_records = parse_single_match(file_path)
        raw_deliveries_list.extend(parsed_records)
    except Exception as error_message:
        print(f"Skipped corrupt file {file_path.name}: {error_message}")

master_df = pd.DataFrame(raw_deliveries_list)


innings_summary_df = (
    master_df.groupby(["batter", "match_id", "batting_team", "year"])
    .agg(
        runs_scored = ("runs", "sum"),
        legal_balls_faced = ("is_wide", lambda series: (~series).sum()),
        was_dismissed = ("dismissed", "max")
    )
    .reset_index()
)

innings_summary_df["strike_rate"] = (
    innings_summary_df["runs_scored"] / innings_summary_df["legal_balls_faced"].replace(0, float('nan')) * 100
).round(2)



career_summary = (
    innings_summary_df.groupby("batter")
    .agg(
        total_matches = ("match_id", "nunique"),
        total_runs = ("runs_scored", "sum"),
        total_balls = ("legal_balls_faced", "sum"),
        total_dismissals = ("was_dismissed", "sum"),
        highest_score = ("runs_scored", "max"),
        debut_year = ("year", "min"),
        final_year = ("year", "max")
    )
    .reset_index()
)


career_summary["overall_strike_rate"] = (
    career_summary["total_runs"] / career_summary["total_balls"].replace(0, float('nan')) * 100
).round(2)

career_summary["batting_average"] = (
    career_summary["total_runs"] / career_summary["total_dismissals"].replace(0, float('nan'))
).round(2)


milestones_df = (
    innings_summary_df.groupby("batter")
    .agg(
        fifties = ("runs_scored", lambda runs: ((runs >= 50) & (runs < 100)).sum()),
        hundreds = ("runs_scored", lambda runs: (runs >= 100).sum())
    )
    .reset_index()
)
career_summary = career_summary.merge(milestones_df, on="batter", how="left")
teams_list_df = (
    innings_summary_df.groupby("batter")["batting_team"]
    .apply(lambda teams: ", ".join(sorted(teams.dropna().unique())))
    .reset_index()
    .rename(columns={"batting_team": "teams_represented"})
)
career_summary = career_summary.merge(teams_list_df, on="batter", how="left")


most_recent_team_df = (
    innings_summary_df.sort_values(["year", "match_id"])
    .groupby("batter")
    .last()
    .reset_index()[["batter", "batting_team"]]
    .rename(columns={"batting_team": "last_known_team"})
)
career_summary = career_summary.merge(most_recent_team_df, on="batter", how="left")


def map_position_to_role(position):
    if position <= 2:   return "Opener"
    elif position == 3: return "No. 3"
    elif position == 4: return "No. 4"
    elif position <= 6: return "Finisher"
    return "Lower Order"

position_tracking_df = (
    master_df.groupby(["batter", "match_id"])["batting_position"]
    .first()
    .reset_index()
)
position_tracking_df["tactical_role"] = position_tracking_df["batting_position"].apply(map_position_to_role)

majority_role_df = (
    position_tracking_df.groupby("batter")["tactical_role"]
    .agg(lambda series: series.value_counts().index[0])
    .reset_index()
    .rename(columns={"tactical_role": "predominant_batting_position"})
)
career_summary = career_summary.merge(majority_role_df, on="batter", how="left")


qualified_filter = (career_summary["total_matches"] >= MIN_MATCHES_REQUIRED) & (career_summary["total_runs"] >= MIN_RUNS_REQUIRED)
career_summary = career_summary[qualified_filter].sort_values("total_runs", ascending=False).reset_index(drop=True)

eligible_batters_set = set(career_summary["batter"].tolist())



yearly_breakdown = (
    innings_summary_df[innings_summary_df["batter"].isin(eligible_batters_set)]
    .groupby(["batter", "year"])
    .agg(
        matches = ("match_id", "nunique"),
        runs = ("runs_scored", "sum"),
        balls = ("legal_balls_faced", "sum"),
        dismissals = ("was_dismissed", "sum"),
        fifties = ("runs_scored", lambda runs: ((runs >= 50) & (runs < 100)).sum()),
        hundreds = ("runs_scored", lambda runs: (runs >= 100).sum())
    )
    .reset_index()
)
yearly_breakdown["strike_rate"] = (yearly_breakdown["runs"] / yearly_breakdown["balls"].replace(0, float('nan')) * 100).round(2)
yearly_breakdown["average"] = (yearly_breakdown["runs"] / yearly_breakdown["dismissals"].replace(0, float('nan'))).round(2)


situational_df = master_df[master_df["batter"].isin(eligible_batters_set)].copy()
situational_df["is_boundary"] = situational_df["runs"].isin([4, 6])
situational_df["is_dot_ball"] = situational_df["runs"] == 0

def aggregate_phase_metrics(group_column_key, prefix_naming_dictionary):
    match_phase_aggregates = (
        situational_df.groupby(["batter", "match_id", group_column_key])
        .agg(
            runs_built = ("runs", "sum"),
            balls_seen = ("is_wide", lambda series: (~series).sum()),
            outs_taken = ("dismissed", "max"),
            boundaries = ("is_boundary", "sum"),
            dots       = ("is_dot_ball", "sum")
        )
        .reset_index()
    )
    
    match_phase_aggregates["sr"] = (match_phase_aggregates["runs_built"] / match_phase_aggregates["balls_seen"].replace(0, float('nan')) * 100).round(2)
    match_phase_aggregates["dot_ratio"] = (match_phase_aggregates["dots"] / match_phase_aggregates["balls_seen"].replace(0, float('nan')) * 100).round(2)

    career_phase_averages = (
        match_phase_aggregates.groupby(["batter", group_column_key])
        .agg(
            total_runs = ("runs_built", "sum"),
            total_balls = ("balls_seen", "sum"),
            total_dismissals = ("outs_taken", "sum"),
            mean_sr = ("sr", "mean"),
            mean_dot_pct = ("dot_ratio", "mean")
        )
        .reset_index()
    )

    flat_unpivoted_result = situational_df[["batter"]].drop_duplicates()
    
    for phase_key_value, output_column_prefix in prefix_naming_dictionary.items():
        subset_slice = career_phase_averages[career_phase_averages[group_column_key] == phase_key_value].copy()
        subset_slice = subset_slice.rename(columns={
            "total_runs": f"{output_column_prefix}_runs",
            "total_balls": f"{output_column_prefix}_balls",
            "total_dismissals": f"{output_column_prefix}_dismissals",
            "mean_sr": f"{output_column_prefix}_sr",
            "mean_dot_pct": f"{output_column_prefix}_dot_pct"
        }).drop(columns=[group_column_key])
        
        flat_unpivoted_result = flat_unpivoted_result.merge(subset_slice, on="batter", how="left")
        
    return flat_unpivoted_result

ball_count_phases_df = aggregate_phase_metrics("ball_phase", {"phase_1_10": "p1_10", "phase_10_40": "p10_40", "phase_40plus": "p40plus"})
over_number_phases_df = aggregate_phase_metrics("over_phase", {"powerplay": "pp", "middle": "mid", "death": "death"})
composite_phases_df = ball_count_phases_df.merge(over_number_phases_df, on="batter", how="outer")



pressure_splits_summary = (
    master_df[master_df["batter"].isin(eligible_batters_set)]
    .groupby(["batter", "is_knockout"])
    .agg(
        appearances = ("match_id", "nunique"),
        runs = ("runs", "sum"),
        balls = ("is_wide", lambda series: (~series).sum()),
        dismissals = ("dismissed", "sum")
    )
    .reset_index()
)
pressure_splits_summary["strike_rate"] = (pressure_splits_summary["runs"] / pressure_splits_summary["balls"].replace(0, float('nan')) * 100).round(2)
pressure_splits_summary["average"] = (pressure_splits_summary["runs"] / pressure_splits_summary["dismissals"].replace(0, float('nan'))).round(2)

def generate_flat_pressure_split(dataframe, knockout_flag, column_prefix):
    filtered_slice = dataframe[dataframe["is_knockout"] == knockout_flag].copy()
    return filtered_slice.rename(columns={
        "appearances": f"{column_prefix}_matches",
        "runs": f"{column_prefix}_runs",
        "balls": f"{column_prefix}_balls",
        "dismissals": f"{column_prefix}_dismissals",
        "strike_rate": f"{column_prefix}_sr",
        "average": f"{column_prefix}_avg"
    }).drop(columns=["is_knockout"])

league_stage_split = generate_flat_pressure_split(pressure_splits_summary, knockout_flag=False, column_prefix="league")
knockout_stage_split = generate_flat_pressure_split(pressure_splits_summary, knockout_flag=True, column_prefix="knockout")
composite_pressure_df = league_stage_split.merge(knockout_stage_split, on="batter", how="outer")


export_destination_root = Path("IPL")

career_summary.to_csv(export_destination_root / "batter_summary.csv", index=False)
yearly_breakdown.to_csv(export_destination_root / "batter_yearly.csv", index=False)
composite_phases_df.to_csv(export_destination_root / "batter_phase.csv", index=False)
composite_pressure_df.to_csv(export_destination_root / "batter_knockout.csv", index=False)

print(f"\nExecution finished! Cleaned outputs exported to '{DATA_DIRECTORY}/'.")

In [ ]:
#bowler file csv extractor
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
import pandas as pd


DATA_DIRECTORY = r"IPL"
MIN_MATCHES_REQUIRED = 5
MIN_WICKETS_REQUIRED = 5


NON_BOWLER_DISMISSALS = {
    "run out", "retired hurt", "obstructing the field", "handled the ball"
}


TEAM_NAME_MAPPING = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Deccan Chargers": "Sunrisers Hyderabad"
}

def standardize_team_name(team_name):
    return TEAM_NAME_MAPPING.get(team_name, team_name)



def parse_single_match(file_path):
    with open(file_path, encoding="utf-8") as file:
        match_data = json.load(file)

    match_info = match_data.get("info", {})
    match_dates = match_info.get("dates", ["1900-01-01"])
    match_year = int(str(match_dates[0])[:4])
    
    competing_teams = [standardize_team_name(team) for team in match_info.get("teams", [])]
    
    event_info = match_info.get("event", {})
    stage_name = event_info.get("stage", "").lower() if isinstance(event_info, dict) else ""
    is_knockout_match = any(keyword in stage_name for keyword in ["final", "semi", "playoff", "eliminator", "qualifier"])

    delivery_records = []

    for inning in match_data.get("innings", []):
        batting_team = standardize_team_name(inning.get("team", ""))
        
        
        bowling_team = ""
        for team in competing_teams:
            if team != batting_team:
                bowling_team = team
                break

        
        consecutive_wickets_count = defaultdict(int)

        for over_data in inning.get("overs", []):
            over_number = over_data.get("over", 0)

            
            if over_number <= 5:
                innings_phase = "powerplay"
            elif over_number <= 14:
                innings_phase = "middle"
            else:
                innings_phase = "death"

            for delivery in over_data.get("deliveries", []):
                bowler_name = delivery.get("bowler", "")
                
                runs_dict = delivery.get("runs", {})
                runs_total = runs_dict.get("total", 0)
                runs_batter = runs_dict.get("batter", 0)
                
                extras_dict = delivery.get("extras", {})
                is_wide_ball = "wides" in extras_dict
                is_noball = "noballs" in extras_dict
                is_legal_ball = not is_wide_ball

                
                bowler_wickets_list = []
                for wicket in delivery.get("wickets", []):
                    dismissal_kind = wicket.get("kind", "")
                    if dismissal_kind not in NON_BOWLER_DISMISSALS:
                        bowler_wickets_list.append(dismissal_kind)
                
                took_wicket = len(bowler_wickets_list) > 0
                is_hat_trick = False

                
                if is_legal_ball:
                    if took_wicket:
                        consecutive_wickets_count[bowler_name] += 1
                        
                        if consecutive_wickets_count[bowler_name] >= 3:
                            is_hat_trick = True
                    else:
                        
                        consecutive_wickets_count[bowler_name] = 0

                delivery_records.append({
                    "match_id": file_path.stem,
                    "year": match_year,
                    "bowling_team": bowling_team,
                    "bowler": bowler_name,
                    "phase": innings_phase,
                    "over_num": over_number,
                    "runs_total": runs_total,
                    "runs_batter": runs_batter,
                    "is_wide": is_wide_ball,
                    "is_noball": is_noball,
                    "is_legal": is_legal_ball,
                    "wickets": len(bowler_wickets_list),
                    "dismissal_kinds": bowler_wickets_list,
                    "is_hat_trick": is_hat_trick,
                    "is_knockout": is_knockout_match,
                })

    return delivery_records



print("Reading and parsing JSON data profiles...")
all_match_files = list(Path(DATA_DIRECTORY).glob("*.json"))
raw_deliveries_list = []
corrupt_files = []

for file_path in all_match_files:
    try:
        raw_deliveries_list.extend(parse_single_match(file_path))
    except Exception as error_message:
        corrupt_files.append((file_path.name, str(error_message)))

master_df = pd.DataFrame(raw_deliveries_list)

print(f" {len(all_match_files)} files parsed | {len(master_df):,} deliveries analyzed")



match_aggregated_df = (
    master_df.groupby(["bowler", "match_id", "bowling_team", "year", "is_knockout"])
    .agg(
        balls_bowled = ("is_legal", "sum"),
        runs_conceded = ("runs_total", "sum"),
        wickets_taken = ("wickets", "sum"),
        hat_tricks_count = ("is_hat_trick", "sum")
    )
    .reset_index()
)

match_aggregated_df["economy"] = (
    match_aggregated_df["runs_conceded"] / (match_aggregated_df["balls_bowled"] / 6).replace(0, np.nan)
).round(2)


career_summary = (
    match_aggregated_df.groupby("bowler")
    .agg(
        total_matches = ("match_id", "nunique"),
        total_balls = ("balls_bowled", "sum"),
        total_runs = ("runs_conceded", "sum"),
        total_wickets = ("wickets_taken", "sum"),
        highest_match_wickets = ("wickets_taken", "max"),
        hat_tricks = ("hat_tricks_count", "sum")
    )
    .reset_index()
)


career_summary["overall_economy"] = (
    career_summary["total_runs"] / (career_summary["total_balls"] / 6).replace(0, np.nan)
).round(2)

career_summary["overall_bowling_sr"] = (
    career_summary["total_balls"] / career_summary["total_wickets"].replace(0, np.nan)
).round(2)

career_summary["average"] = (
    career_summary["total_runs"] / career_summary["total_wickets"].replace(0, np.nan)
).round(2)


seasonal_wickets = match_aggregated_df.groupby(["bowler", "year"])["wickets_taken"].sum().reset_index()
best_season_idx = seasonal_wickets.groupby("bowler")["wickets_taken"].idxmax()
best_season_df = seasonal_wickets.loc[best_season_idx, ["bowler", "year"]].rename(columns={"year": "best_season"})
career_summary = career_summary.merge(best_season_df, on="bowler", how="left")


hauls_df = (
    match_aggregated_df.groupby("bowler")
    .agg(
        three_wicket_hauls = ("wickets_taken", lambda x: (x >= 3).sum()),
        five_wicket_hauls  = ("wickets_taken", lambda x: (x >= 5).sum())
    )
    .reset_index()
)
career_summary = career_summary.merge(hauls_df, on="bowler", how="left")


dismissal_items = []
for idx, row in master_df[master_df["wickets"] > 0].iterrows():
    for method in row["dismissal_kinds"]:
        dismissal_items.append({"bowler": row["bowler"], "method": method})

if dismissal_items:
    dismissals_master_df = pd.DataFrame(dismissal_items)
else:
    dismissals_master_df = pd.DataFrame(columns=["bowler", "method"])

target_dismissal_styles = ["bowled", "caught", "lbw", "stumped"]
for style in target_dismissal_styles:
    style_counts = (
        dismissals_master_df[dismissals_master_df["method"] == style]
        .groupby("bowler")
        .size()
        .reset_index(name=style)
    )
    career_summary = career_summary.merge(style_counts, on="bowler", how="left")

career_summary[target_dismissal_styles] = career_summary[target_dismissal_styles].fillna(0).astype(int)


def find_prominent_method(row):
    style_scores = {style: row[style] for style in target_dismissal_styles}
    return max(style_scores, key=style_scores.get)

career_summary["most_dismissals"] = career_summary.apply(find_prominent_method, axis=1)

teams_string_df = (
    match_aggregated_df.groupby("bowler")["bowling_team"]
    .apply(lambda teams: ", ".join(sorted(teams.dropna().unique())))
    .reset_index()
    .rename(columns={"bowling_team": "teams_represented"})
)
career_summary = career_summary.merge(teams_string_df, on="bowler", how="left")

last_known_team_df = (
    match_aggregated_df.sort_values("year")
    .groupby("bowler")
    .last()
    .reset_index()[["bowler", "bowling_team"]]
    .rename(columns={"bowling_team": "last_team"})
)
career_summary = career_summary.merge(last_known_team_df, on="bowler", how="left")


qualification_mask = (career_summary["total_matches"] >= MIN_MATCHES_REQUIRED) & (career_summary["total_wickets"] >= MIN_WICKETS_REQUIRED)
career_summary = career_summary[qualification_mask].sort_values("total_wickets", ascending=False).reset_index(drop=True)


career_summary = career_summary[[
    "bowler", "total_matches", "total_balls", "total_runs",
    "total_wickets", "overall_economy", "overall_bowling_sr", "average",
    "highest_match_wickets", "best_season",
    "three_wicket_hauls", "five_wicket_hauls", "hat_tricks",
    "most_dismissals", "bowled", "caught", "lbw", "stumped",
    "teams_represented", "last_team",
]]

eligible_bowlers_set = set(career_summary["bowler"].tolist())



yearly_breakdown = (
    match_aggregated_df[match_aggregated_df["bowler"].isin(eligible_bowlers_set)]
    .groupby(["bowler", "year"])
    .agg(
        matches = ("match_id", "nunique"),
        balls   = ("balls_bowled", "sum"),
        runs    = ("runs_conceded", "sum"),
        wickets = ("wickets_taken", "sum")
    )
    .reset_index()
)
yearly_breakdown["economy"] = (yearly_breakdown["runs"] / (yearly_breakdown["balls"] / 6).replace(0, np.nan)).round(2)
yearly_breakdown["bowling_sr"] = (yearly_breakdown["balls"] / yearly_breakdown["wickets"].replace(0, np.nan)).round(2)


legal_phases_df = master_df[master_df["bowler"].isin(eligible_bowlers_set) & master_df["is_legal"]].copy()
legal_phases_df["is_dot_ball"] = legal_phases_df["runs_batter"] == 0
legal_phases_df["is_boundary_hit"] = legal_phases_df["runs_batter"].isin([4, 6])

match_phase_groups = (
    legal_phases_df.groupby(["bowler", "match_id", "phase"])
    .agg(
        balls_seen  = ("is_legal", "sum"),
        runs_given  = ("runs_total", "sum"),
        outs_earned = ("wickets", "sum"),
        dots_count  = ("is_dot_ball", "sum"),
        boundaries  = ("is_boundary_hit", "sum")
    )
    .reset_index()
)
match_phase_groups["eco"] = (match_phase_groups["runs_given"] / (match_phase_groups["balls_seen"] / 6).replace(0, np.nan)).round(2)
match_phase_groups["dot_pct"] = (match_phase_groups["dots_count"] / match_phase_groups["balls_seen"].replace(0, np.nan) * 100).round(2)
match_phase_groups["bound_pct"] = (match_phase_groups["boundaries"] / match_phase_groups["balls_seen"].replace(0, np.nan) * 100).round(2)

career_phase_averages = (
    match_phase_groups.groupby(["bowler", "phase"])
    .agg(
        sum_balls_bowled = ("balls_seen", "sum"),
        sum_runs_conceded = ("runs_given", "sum"),
        sum_wickets_taken = ("outs_earned", "sum"),
        mean_dot_ratio = ("dot_pct", "mean"),
        mean_boundary_ratio = ("bound_pct", "mean")
    )
    .reset_index()
)
career_phase_averages["overall_eco"] = (career_phase_averages["sum_runs_conceded"] / (career_phase_averages["sum_balls_bowled"] / 6).replace(0, np.nan)).round(2)
career_phase_averages["overall_sr"] = (career_phase_averages["sum_balls_bowled"] / career_phase_averages["sum_wickets_taken"].replace(0, np.nan)).round(2)

composite_phases_df = pd.DataFrame({"bowler": list(eligible_bowlers_set)})
defined_t20_phases = ["powerplay", "middle", "death"]

for active_phase in defined_t20_phases:
    phase_slice = career_phase_averages[career_phase_averages["phase"] == active_phase].copy()
    phase_slice = phase_slice.rename(columns={
        "sum_balls_bowled": f"{active_phase}_balls",
        "sum_runs_conceded": f"{active_phase}_runs",
        "sum_wickets_taken": f"{active_phase}_wickets",
        "overall_eco": f"{active_phase}_economy",
        "overall_sr": f"{active_phase}_bowling_sr",
        "mean_dot_ratio": f"{active_phase}_dot_pct",
        "mean_boundary_ratio": f"{active_phase}_boundary_pct",
    }).drop(columns=["phase"]) 
    
    composite_phases_df = composite_phases_df.merge(phase_slice, on="bowler", how="left")


pressure_splits_summary = (
    match_aggregated_df[match_aggregated_df["bowler"].isin(eligible_bowlers_set)]
    .groupby(["bowler", "is_knockout"])
    .agg(
        appearances = ("match_id", "nunique"),
        balls   = ("balls_bowled", "sum"),
        runs    = ("runs_conceded", "sum"),
        wickets = ("wickets_taken", "sum")
    )
    .reset_index()
)
pressure_splits_summary["economy"] = (pressure_splits_summary["runs"] / (pressure_splits_summary["balls"] / 6).replace(0, np.nan)).round(2)
pressure_splits_summary["bowling_sr"] = (pressure_splits_summary["balls"] / pressure_splits_summary["wickets"].replace(0, np.nan)).round(2)
pressure_splits_summary["average"] = (pressure_splits_summary["runs"] / pressure_splits_summary["wickets"].replace(0, np.nan)).round(2)

def generate_flat_pressure_split(dataframe, knockout_flag, column_prefix):
    filtered_slice = dataframe[dataframe["is_knockout"] == knockout_flag].copy()
    return filtered_slice.rename(columns={
        "appearances": f"{column_prefix}_matches",
        "balls": f"{column_prefix}_balls",
        "runs": f"{column_prefix}_runs",
        "wickets": f"{column_prefix}_wickets",
        "economy": f"{column_prefix}_economy",
        "bowling_sr": f"{column_prefix}_bowling_sr",
        "average": f"{column_prefix}_average"
    }).drop(columns=["is_knockout"])

league_stage_split = generate_flat_pressure_split(pressure_splits_summary, knockout_flag=False, column_prefix="league")
knockout_stage_split = generate_flat_pressure_split(pressure_splits_summary, knockout_flag=True, column_prefix="knockout")
composite_pressure_df = league_stage_split.merge(knockout_stage_split, on="bowler", how="outer")



export_destination_root = Path(DATA_DIRECTORY)

career_summary.to_csv(export_destination_root / "bowler_summary.csv", index=False)
yearly_breakdown.to_csv(export_destination_root / "bowler_yearly.csv", index=False)
composite_phases_df.to_csv(export_destination_root / "bowler_phase.csv", index=False)
composite_pressure_df.to_csv(export_destination_root / "bowler_knockout.csv", index=False)

print(f"\nExecution finished! Cleaned outputs exported to '{DATA_DIRECTORY}/'.")